# Simple RAG Training Pipeline (Cgft)

This notebook follows the same 4-step flow and uses the new CgftPipeline API:

1. **Point to dataset** - Chunk and upload documents
2. **Create synthetic QA** - Generate question-answer pairs with CgftPipeline
3. **Create env** - Load search environment
4. **Run training job** - Launch the training


## Setup

Install dependencies and configure API credentials.


In [1]:
# For Google Colab - clone repo and install
# !git clone https://github.com/cgftinc/synthetic-data-prep-lib.git
# %cd synthetic-data-prep-lib
# %pip install -e .[dev]

In [2]:
# Local development setup
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

cwd = Path.cwd()
candidate_roots = [cwd, cwd.parent]
repo_root = next((p for p in candidate_roots if (p / "src" / "synthetic_data_prep").exists()), cwd)
src_path = repo_root / "src"
if src_path.exists() and str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


In [3]:
from synthetic_data_prep.utils import ensure_safe_python_version

ensure_safe_python_version()

In [4]:
# Configuration

# create an API key from the cgft console (https://app.cgft.io/account/api-keys).
API_KEY = "sk_RZEmogqiKirhhDCtipwhSKeqFqnuLUjoxfTQqTkRvNWyuMFOoJkEJxWgJnwoobKS"
BASE_URL = "https://app.cgft.io"
LLM_API_KEY = "EldWZtATVrvUQMKTjgAQOHeadaoiHJoD8SrorScBm6IUn44yARevJQQJ99CBACHYHv6XJ3w3AAAAACOGUUWL"
LLM_BASE_URL = "https://expt-platform-foundry.openai.azure.com/openai/v1"

# Dataset configuration
DOCS_PATH = "./samples/posthog/"
CORPUS_NAME = "posthog"

# Corpus intent used for corpus profiling (summary + example queries)
CORPUS_DESCRIPTION = "Posthog documentation including docs, company policy, etc."
EXAMPLE_QUERIES = [
    "how to feature flag",
    "setup reverse proxy to avoid ad blockers",
    "posthog install nextjs",
]

# QA generation configuration (CgftPipeline)
TOTAL_SAMPLES = 100
OUTPUT_DIR = "outputs/cgft_simple_rag"

# Optional: name for your training experiment
EXPERIMENT_NAME = "posthog search v1"

# Generation/filter/refinement behavior follows library defaults.
# Override in the config cell below only when you intentionally want non-default behavior.

## Step 1: Point to Dataset

Chunk your documents and upload to corpus API in one line.


In [5]:
from synthetic_data_prep.corpus.corpora.source import CorporaChunkSource

source = CorporaChunkSource(api_key=API_KEY, corpus_name=CORPUS_NAME, base_url=BASE_URL)
source.populate_from_folder(DOCS_PATH)

Chunking documents from ./samples/posthog/...
ChunkCollection Summary
Total chunks: 3225
Total files: 1183

Chunk Size Statistics:
  Min: 29 chars (contribute/badge.md, chunk 0)
  Max: 2048 chars (data-warehouse/query.mdx, chunk 0)
  Mean: 1148 chars

File Structure:
├── data-warehouse/
│   ├── sql/
│   │   ├── index.mdx (8 chunks)
│   │   ├── useful-functions.mdx (7 chunks)
│   │   ├── variables.mdx (2 chunks)
│   │   └── data-access.mdx (3 chunks)
│   ├── views/
│   │   ├── index.mdx (2 chunks)
│   │   └── materialize.mdx (2 chunks)
│   ├── _snippets/
│   │   └── query-join-example.mdx (1 chunks)
│   ├── sources/
│   │   ├── stripe.mdx (1 chunks)
│   │   ├── tiktok-ads.mdx (1 chunks)
│   │   ├── azure-blob.mdx (1 chunks)
│   │   ├── attio.mdx (1 chunks)
│   │       ... 31 more files
│   ├── under-the-hood.md (1 chunks)
│   ├── query.mdx (4 chunks)
│   ├── tutorials.mdx (1 chunks)
│   ├── changelog.mdx (1 chunks)
│       ... 5 more files
├── how-posthog-works/
│   ├── clickhouse.md (3

Uploading chunks:   0%|          | 0/33 [00:00<?, ?batch/s]


Upload complete! Inserted: 3225


## Step 2: Create Synthetic QA

Generate synthetic QA pairs with `CgftPipeline` using the corpus from Step 1.


In [6]:
from synthetic_data_prep.qa_generation.cgft_models import (
    CgftPipelineConfig,
    CorpusConfig,
    CorpusContextConfig,
    EntityExtractionLLMConfig,
    FilteringConfig,
    GenerationConfig,
    GroundingLLMFilterConfig,
    LinkerConfig,
    LLMDirectGenerationConfig,
    LLMEnvGenerationConfig,
    LLMGuidedLinkerConfig,
    OutputConfig,
    PlatformConfig,
    RefinementConfig,
    RetrievalLLMFilterConfig,
    TargetsConfig,
    TransformationConfig,
)
from synthetic_data_prep.qa_generation.cgft_pipeline import CgftPipeline

LLM_MODEL = "gpt-5.4"

# --- Batch / concurrency settings ---
# max_concurrent controls how many LLM calls run in parallel within each stage.
# batch_enabled=True  → parallel LLM calls (default, recommended for > ~5 samples)
# batch_enabled=False → sequential calls (useful for debugging a single request)
MAX_CONCURRENT = 8

cfg = CgftPipelineConfig(
    platform=PlatformConfig(api_key=API_KEY, base_url=BASE_URL, llm_api_key=LLM_API_KEY, llm_base_url=LLM_BASE_URL),
    corpus=CorpusConfig(corpus_name=CORPUS_NAME, min_chunk_chars=400),
    corpus_context=CorpusContextConfig(
        description=CORPUS_DESCRIPTION,
        example_queries=EXAMPLE_QUERIES,
        model=LLM_MODEL,
        num_top_level_samples=4,
        num_random_samples=4,
        min_chunk_chars=400,
        # generate_entity_patterns=True (default): LLM generates entity/code/domain patterns
        # from corpus samples during profiling, improving BM25 chunk linking quality.
        # Set to False to skip the extra LLM call and fall back to metadata-based BM25 queries.
        entity_extraction_llm=EntityExtractionLLMConfig(model=LLM_MODEL)
    ),
    linker=LinkerConfig(
        llm_guided=LLMGuidedLinkerConfig(model=LLM_MODEL),
    ),
    generation=GenerationConfig(
        llm_direct=LLMDirectGenerationConfig(
            model=LLM_MODEL,
            # Batch: all generation LLM calls run concurrently (up to MAX_CONCURRENT at a time).
            # Each task gets its own per-prompt system prompt, so per-item context is preserved.
            max_concurrent=MAX_CONCURRENT,
            batch_enabled=True,
            show_batch_progress=True,
        ),
        llm_env=LLMEnvGenerationConfig(model=LLM_MODEL),
    ),
    transformation=TransformationConfig(model=LLM_MODEL, validation_model=LLM_MODEL),
    filtering=FilteringConfig(
        retrieval_llm=RetrievalLLMFilterConfig(
            judge_model=LLM_MODEL,
            # Batch: search stays sequential (BM25 per-item), judge calls run concurrently.
            # Items that exceed the overlap pre-gate threshold skip the judge entirely.
            max_concurrent=MAX_CONCURRENT,
            batch_enabled=True,
        ),
        grounding_llm=GroundingLLMFilterConfig(
            judge_model=LLM_MODEL,
            # Batch: search stays sequential, grounding judge calls run concurrently.
            max_concurrent=MAX_CONCURRENT,
            batch_enabled=True,
        ),
    ),
    refinement=RefinementConfig(model=LLM_MODEL),
    targets=TargetsConfig(total_samples=TOTAL_SAMPLES),
    output=OutputConfig(dir=OUTPUT_DIR, train_jsonl="train.jsonl", eval_jsonl="eval.jsonl"),
)

cfg.resolve_api_keys()
print("generator model:", cfg.generation.llm_direct.model)
print("max_same_seed_attempts_before_reanchor:", cfg.refinement.max_same_seed_attempts_before_reanchor)
print("filter chain:", cfg.filtering.filters)
print("generate_entity_patterns:", cfg.corpus_context.generate_entity_patterns)
print()
print("Batch settings")
print(f"  generation  → batch_enabled={cfg.generation.llm_direct.batch_enabled}, max_concurrent={cfg.generation.llm_direct.max_concurrent}, show_progress={cfg.generation.llm_direct.show_batch_progress}")
print(f"  grounding   → batch_enabled={cfg.filtering.grounding_llm.batch_enabled}, max_concurrent={cfg.filtering.grounding_llm.max_concurrent}")
print(f"  retrieval   → batch_enabled={cfg.filtering.retrieval_llm.batch_enabled}, max_concurrent={cfg.filtering.retrieval_llm.max_concurrent}")

generator model: gpt-5.4
max_same_seed_attempts_before_reanchor: 3
filter chain: ['grounding_llm', 'retrieval_too_easy_llm']
generate_entity_patterns: True

Batch settings
  generation  → batch_enabled=True, max_concurrent=8, show_progress=True
  grounding   → batch_enabled=True, max_concurrent=8
  retrieval   → batch_enabled=True, max_concurrent=8


### Batch / parallel processing

With `batch_enabled=True` (the default), LLM calls within each pipeline stage run concurrently up to `max_concurrent`. Stages themselves still run in order:

```
Generator.generate(tasks)
    ├─ prepare all tasks sequentially (linker.link per task)
    └─ batch LLM call  ← parallel, up to max_concurrent

GroundingFilter.evaluate(items)
    ├─ BM25 search sequentially (one per item)
    └─ batch judge calls  ← parallel, up to max_concurrent

RetrievalFilter.evaluate(items)
    ├─ BM25 search sequentially
    ├─ items that hit the overlap pre-gate → verdict immediately (no judge call)
    └─ remaining items → batch judge calls  ← parallel
```

Expected speedup vs sequential (`max_concurrent=1`): **3–6× faster** for batches of ≥ 20 items, limited by LLM latency rather than call count.

To debug a single item, set `batch_enabled=False` in any of the three configs above.

In [7]:
# Reuse the already-loaded corpus source from Step 1.
import importlib
from pprint import pprint

import synthetic_data_prep.qa_generation.cgft_pipeline as _cgft_pipeline_mod
import synthetic_data_prep.qa_generation.generators.direct_llm as _direct_llm_mod

# Force-refresh generation modules in case notebook state still has stale classes.
importlib.reload(_direct_llm_mod)
importlib.reload(_cgft_pipeline_mod)
CgftPipeline = _cgft_pipeline_mod.CgftPipeline

cgft = CgftPipeline(cfg, source_factory=lambda _cfg: source)
result = cgft.run()

train_data = result["train_dataset"]
eval_data = result["eval_dataset"]
stats = result["stats"]

# Show entity extraction patterns generated during corpus profiling.
# These drive BM25 query generation in the structural linker, replacing brittle
# metadata-based queries (headers/first sentence) with corpus-specific entities.
entity_extraction = cfg.corpus_context.entity_extraction
if entity_extraction is not None:
    print("Entity extraction patterns (confidence=%s)" % entity_extraction.confidence)
    pprint(
        {
            "entity_names": entity_extraction.entity_names,
            "domain_terms": entity_extraction.domain_terms,
            "code_patterns": entity_extraction.code_patterns,
            "query_templates": entity_extraction.query_templates,
        }
    )
else:
    print("Entity extraction patterns: not generated (generate_entity_patterns=False or profiling disabled)")

print()
print("Run summary")
pprint(
    {
        "raw_candidates_total": stats.get("raw_candidates_total"),
        "passed_total": stats.get("passed_total"),
        "rejected_total": stats.get("rejected_total"),
        "regenerated_total": stats.get("regenerated_total"),
        "round_limit": stats.get("round_limit"),
        "filter_chain": stats.get("filter_chain"),
    }
)

retrieval_stats = stats.get("retrieval_too_easy_filter") or {}
if retrieval_stats:
    print()
    print("Retrieval filter diagnostics")
    pprint(
        {
            "passed": retrieval_stats.get("passed"),
            "needs_refinement": retrieval_stats.get("needs_refinement"),
            "rejected": retrieval_stats.get("rejected"),
            "errors": retrieval_stats.get("errors"),
            "judge_calls": retrieval_stats.get("judge_calls"),
            "judge_answerable_true": retrieval_stats.get("judge_answerable_true"),
            "judge_answerable_false": retrieval_stats.get("judge_answerable_false"),
            "too_easy_total": retrieval_stats.get("too_easy_total"),
            "too_easy_due_to_overlap_pre_gate": retrieval_stats.get("too_easy_due_to_overlap_pre_gate"),
            "too_easy_due_to_judge": retrieval_stats.get("too_easy_due_to_judge"),
            "overlap_threshold_triggered": retrieval_stats.get("overlap_threshold_triggered"),
            "too_easy_overlap_threshold_triggered": retrieval_stats.get("too_easy_overlap_threshold_triggered"),
        }
    )
    print("judge_reason_tags:")
    pprint(retrieval_stats.get("judge_reason_tags", {}))
else:
    print()
    print("Retrieval (too-easy) filter diagnostics are unavailable in result['stats'].")

grounding_stats = stats.get("grounding_filter", {})
if grounding_stats:
    print()
    print("Grounding filter diagnostics")
    pprint(
        {
            "passed": grounding_stats.get("passed"),
            "needs_refinement": grounding_stats.get("needs_refinement"),
            "rejected": grounding_stats.get("rejected"),
            "errors": grounding_stats.get("errors"),
            "judge_answerable_true": grounding_stats.get("judge_answerable_true"),
            "judge_answerable_false": grounding_stats.get("judge_answerable_false"),
            "unsupported_total": grounding_stats.get("unsupported_total"),
        }
    )

transformation_stats = stats.get("transformation", {})
if transformation_stats:
    print()
    print("Transformation diagnostics")
    pprint(transformation_stats)

stats


Skipping invalid entity extraction regex 'property_keys_in_objects': (?<=[{,]\s*)([A-Za-z_][\w]*)\s*:


Processing prompts:   0%|          | 0/100 [00:00<?, ?it/s]

Skipping task task_00042: batch LLM call failed.
Skipping task task_00022: batch LLM call failed.
Skipping task task_00071: batch LLM call failed.


Processing prompts:   0%|          | 0/9 [00:00<?, ?it/s]

Skipping task task_00065__regen_01: batch LLM call failed.


LLMSanityValidator call failed.
Traceback (most recent call last):
  File "/home/angelpan/git/cgft/synthetic-data-prep-lib/src/synthetic_data_prep/qa_generation/transformers/validator.py", line 65, in is_valid
    response = self.client.chat.completions.create(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/angelpan/git/cgft/synthetic-data-prep-lib/.venv/lib/python3.12/site-packages/openai/_utils/_utils.py", line 286, in wrapper
    return func(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^
  File "/home/angelpan/git/cgft/synthetic-data-prep-lib/.venv/lib/python3.12/site-packages/openai/resources/chat/completions/completions.py", line 1204, in create
    return self._post(
           ^^^^^^^^^^^
  File "/home/angelpan/git/cgft/synthetic-data-prep-lib/.venv/lib/python3.12/site-packages/openai/_base_client.py", line 1297, in post
    return cast(ResponseT, self.request(cast_to, opts, stream=stream, stream_cls=stream_cls))
                           ^^^^^^^^^^^^^^^^^

Entity extraction patterns (confidence=high)
{'code_patterns': {'config_keys': '\\b(?:api_host|persistence|cookie_persisted_properties|SECURE_COOKIES|DEBUG|TEST)\\b',
                   'docs_links': '\\[[^\\]]+\\]\\((?:/docs|/manual|https?://)[^)]+\\)',
                   'env_var_assignments': '\\bSECURE_COOKIES\\s*=\\s*(?:True|False|true|false)',
                   'exported_hook_names': '\\bprocessEvent\\b',
                   'generic_function_calls': '\\b[A-Za-z_$][\\w$]*\\s*\\(',
                   'glossary_terms': '^####\\s+.+$',
                   'javascript_function_calls': '\\b(?:posthog|[A-Za-z_$][\\w$]*)\\.(?:init|register|get_property)\\s*\\(',
                   'jsx_component_tags': '<[A-Z][A-Za-z0-9_.]*(?:\\s+[^>]*)?\\s*/?>',
                   'markdown_headings': '^#{2,4}\\s+.+$',
                   'mdx_import_paths': 'import\\s+[A-Za-z_$][\\w$]*\\s+from\\s+["\'][^"\']+\\.mdx["\']',
                   'persistence_modes': '\\blocalStorage\\+cookie\\b',
           

{'total': 95,
 'train': 74,
 'eval': 21,
 'train_ratio': 0.8,
 'stratify_by': ['qa_type', 'style'],
 'raw_candidates_total': 97,
 'passed_total': 95,
 'rejected_total': 2,
 'regenerated_total': 10,
 'round_limit': 4,
 'filter_chain': ['grounding_llm', 'retrieval_too_easy_llm'],
 'deterministic_guards': {'passed': 105, 'rejected': 1},
 'retrieval_too_easy_filter': {'passed': 95,
  'needs_refinement': 0,
  'rejected': 0,
  'errors': 0,
  'judge_calls': 95,
  'judge_answerable_true': 29,
  'judge_answerable_false': 66,
  'judge_reason_tags': {'too_easy_lexical': 0,
   'unsupported': 3,
   'challenging_answerable_pass': 0,
   'unknown': 92},
  'too_easy_total': 0,
  'too_easy_due_to_overlap_pre_gate': 0,
  'too_easy_due_to_judge': 0,
  'overlap_threshold_triggered': 1,
  'too_easy_overlap_threshold_triggered': 0},
 'grounding_filter': {'passed': 95,
  'needs_refinement': 10,
  'rejected': 0,
  'errors': 0,
  'judge_answerable_true': 95,
  'judge_answerable_false': 10,
  'unsupported_total'

In [8]:
for qa_pair in result['train_dataset'][:10]:
    print("Q:", qa_pair['question'])
    print("A:", qa_pair['answer'])

Q: For a PH workflow that only sends notifications via Slack and SMS, what channel-specific config is required for Slack vs. SMS (e.g., Slack connection, Twilio SID/Auth Token), is a reusable message template needed in this Slack+SMS-only setup, and if a community post about the setup might violate rules, should I contact the mods at hey@posthog.com?
A: For Slack, you need to add a Slack connection so messages can be delivered to a Slack channel. For SMS, you need to configure Twilio with an Account SID and an Auth Token. If you are only using Slack and SMS and not email, you do not need to create a reusable template. If you're unsure whether your community post is allowed, contact the community moderators at hey@posthog.com.
Q: How should I set this up for an ecommerce site where each product page has a unique ID in teh URL: (1) treat all `/products/{id}` pageviews as a single step in Paths, (2) view one combined heatmap for all those product pages, and (3) write a SQL query that retu

## Step 3: Launch Training

Upload datasets and environment, then launch the training job.

In [ ]:
import synthetic_data_prep
from synthetic_data_prep.envs.cgft_search_env import CgftSearchEnv
from synthetic_data_prep.trainer.pipeline import train
experiment_id = train(
    env_class=CgftSearchEnv,
    env_args={
        "api_key": API_KEY,
        "corpus_id": source.corpus_id,
        "base_url": BASE_URL,
    },
    train_dataset=train_data,
    eval_dataset=eval_data,
    prefix="cgft-search",
    api_key=API_KEY,
    base_url=BASE_URL,
    local_modules=[synthetic_data_prep],
    experiment_name=EXPERIMENT_NAME,
)


## Monitoring Training: What to Expect

### Early Training Noise

**Early training**: At the start of a training run, rewards will fluctuate and metrics may be noisy. This is completely normal - the model is still learning basic patterns and its outputs are unstable. Give it some time (usually a few dozen steps) and the signal will clean up and you'll start seeing clear trends.

**Exploration before exploitation**: Ideally, you want to see pass@k climbing first before average reward increases significantly. This means your model is exploring the solution space and learning to solve more prompts (breadth) before it optimizes for higher rewards on those prompts (depth). If average reward shoots up while pass@k stays low, you're likely exploiting a narrow set of easy prompts rather than building robust capabilities.

**Healthy training trajectory**: In a well-configured training run, expect pass@k to increase early as the model learns to solve more distinct prompts. Average reward should follow but lag behind. Eventually both should plateau as the model saturates your training distribution.

**Warning signs**:

- pass@k flat while average reward increases → model is overfitting to a narrow subset of prompts
- both metrics stagnate early → training distribution may be too hard, reward signal too sparse
